# Week 11, Day 5: Filtering and the convolution theorem

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w011/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Check the theorem

Write `circular_convolve(x, h)` with a double loop, `via_fft(x, h)` using the theorem, and `fast_convolve(x, h)` that pads both to length len(x) + len(h) − 1 and uses the FFT route to reproduce `np.convolve`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def circular_convolve(x, h):
    N = len(x)
    return np.array([sum(x[m] * h[(n - m) % N] for m in range(N)) for n in range(N)])

def via_fft(x, h):
    return ...

def fast_convolve(x, h):
    L = len(x) + len(h) - 1
    return ...

r = np.random.default_rng(0); a, b = r.normal(size=8), r.normal(size=8)
print(np.round(circular_convolve(a, b), 4)); print(np.round(via_fft(a, b), 4))

In [ ]:
import numpy as np
def test_circ():
    "circular_convolve wraps around the ends"
    assert np.allclose(circular_convolve(np.array([1.0, 2, 3, 4]), np.array([0.0, 1, 0, 0])), [4, 1, 2, 3])

def test_theorem():
    "the DFT route gives the same answer"
    r = np.random.default_rng(3); a, b = r.normal(size=32), r.normal(size=32)
    assert np.allclose(circular_convolve(a, b), via_fft(a, b))

def test_linear():
    "zero padding turns the FFT route into ordinary convolution"
    r = np.random.default_rng(4); a, b = r.normal(size=20), r.normal(size=7)
    assert np.allclose(fast_convolve(a, b), np.convolve(a, b))

test_circ()
test_theorem()
test_linear()
print("All checks passed")

### Exercise: Measure the membrane's frequency response

Simulate the membrane (τ = 20 ms, Euler steps of 0.05 ms) driven by a unit-amplitude sinusoidal input at each frequency in `test_freqs` for 1 s, and measure the output amplitude as half the range of V over the last 0.5 s. Store the gains in `gains` and compare with the formula.

In [ ]:
tau, dt = 0.020, 0.00005
test_freqs = [2, 8, 32, 100]
gains = []
for f in test_freqs:
    t = np.arange(0, 1, dt)
    V = np.zeros(len(t))
    for k in range(len(t) - 1):
        V[k + 1] = ...
    late = V[len(t) // 2:]
    gains.append(...)

fgrid = np.logspace(0, 2.5, 100)
plt.figure(figsize=(5, 3)); plt.loglog(fgrid, 1 / np.sqrt(1 + (2 * np.pi * fgrid * tau) ** 2), label="formula"); plt.loglog(test_freqs, gains, "o", label="simulation")
plt.xlabel("frequency (Hz)"); plt.ylabel("gain"); plt.legend(frameon=False); plt.show()
print("gains:", np.round(gains, 3), "  ratio of 2 Hz to 100 Hz:", round(gains[0] / gains[3], 1))

In [ ]:
import numpy as np
def test_gain():
    "measured gains match the formula within 5%"
    for f, g in zip(test_freqs, gains):
        assert abs(g - 1 / np.sqrt(1 + (2 * np.pi * f * tau) ** 2)) < 0.05 * max(g, 0.02), (f, g)

test_gain()
print("All checks passed")

### Exercise: A band-pass filter in the frequency domain

Write `bandpass(x, fs, lo, hi, edge=1.0)`: take `rfft`, build a mask that is 1 between lo and hi and falls off outside as a Gaussian of width `edge` Hz, multiply, and transform back with `irfft`. Apply it to a noisy theta signal and to pure white noise, and compare.

In [ ]:
def bandpass(x, fs, lo, hi, edge=1.0):
    X = np.fft.rfft(x)
    f = np.fft.rfftfreq(len(x), 1 / fs)
    dist = np.maximum(lo - f, 0) + np.maximum(f - hi, 0)    # how far each frequency lies outside the band
    mask = ...
    return ...

rng = np.random.default_rng(1993)
fs = 1000
t = np.arange(0, 4, 1 / fs)
lfp = np.sin(2 * np.pi * 8 * t) + 1.5 * rng.normal(size=len(t))
noise = 1.5 * rng.normal(size=len(t))
fig, ax = plt.subplots(2, 1, figsize=(7, 3.6), sharex=True)
ax[0].plot(t, lfp, color="0.75", lw=0.5); ax[0].plot(t, bandpass(lfp, fs, 6, 10), lw=1.5); ax[0].set_title("theta plus noise, filtered 6 to 10 Hz")
ax[1].plot(t, noise, color="0.75", lw=0.5); ax[1].plot(t, bandpass(noise, fs, 6, 10), lw=1.5); ax[1].set_title("pure noise, filtered 6 to 10 Hz")
ax[1].set_xlim(0, 1.5); ax[1].set_xlabel("time (s)"); plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
def test_pass():
    "a sinusoid inside the band passes almost unchanged"
    t = np.arange(4 * 1000) / 1000
    s = np.sin(2 * np.pi * 8 * t)
    assert np.allclose(bandpass(s, 1000, 6, 10), s, atol=0.02)

def test_stop():
    "a sinusoid far outside the band is removed"
    t = np.arange(4 * 1000) / 1000
    assert np.max(np.abs(bandpass(np.sin(2 * np.pi * 40 * t), 1000, 6, 10))) < 0.01

test_pass()
test_stop()
print("All checks passed")

### Exercise: Which phase of theta does this neuron prefer?

Write `analytic_signal(x)` via the FFT (keep bin 0, double bins 1 to N/2 − 1, keep bin N/2 for even N, zero the rest). Compute theta phase from the band-passed LFP, take the phase at each spike, and store the mean resultant length `R` and preferred phase `pref`. As a control, jitter each spike independently by a random amount up to one theta cycle (125 ms) and recompute `R_shuffled`.

In [ ]:
def analytic_signal(x):
    N = len(x)
    X = np.fft.fft(x)
    h = np.zeros(N)
    h[0] = 1
    h[1:N // 2] = 2
    if N % 2 == 0:
        h[N // 2] = 1
    return ...

theta = bandpass(lfp, fs, 6, 10)
phase = np.angle(analytic_signal(theta))
true_phase = np.angle(np.exp(1j * (2 * np.pi * 8 * t - np.pi / 2)))   # phase of the clean 8 Hz sine
# a neuron that fires most near the theta trough (phase pi), about 30 spikes per second
p_spike = 0.03 * np.exp(1.5 * np.cos(true_phase - np.pi)) / 1.65
spike_idx = np.nonzero(rng.random(len(t)) < p_spike)[0]

zbar = ...
R, pref = np.abs(zbar), np.angle(zbar)
shifted = (spike_idx + rng.integers(0, 125, size=len(spike_idx))) % len(t)   # jitter each spike by up to one theta cycle
R_shuffled = ...

plt.figure(figsize=(4, 3)); plt.hist(np.degrees(phase[spike_idx]) % 360, bins=18); plt.xlabel("theta phase at spike (deg)"); plt.ylabel("spikes"); plt.show()
print(f"{len(spike_idx)} spikes;  R = {R:.2f}, preferred phase {np.degrees(pref) % 360:.0f} deg;  shifted control R = {R_shuffled:.3f}")

In [ ]:
import numpy as np
def test_analytic():
    "analytic_signal turns cos into exp(i omega t)"
    t = np.arange(1000) / 1000
    z = analytic_signal(np.cos(2 * np.pi * 10 * t))
    assert np.allclose(z, np.exp(2j * np.pi * 10 * t), atol=1e-9)

def test_locking():
    "the neuron is locked near the theta trough (pi), with R above 0.3"
    assert abs(np.angle(np.exp(1j * (pref - np.pi)))) < 0.4 and R > 0.3

def test_control():
    "spikes jittered independently by up to one cycle show far weaker locking"
    assert R_shuffled < R / 3

test_analytic()
test_locking()
test_control()
print("All checks passed")